Notebook for RS/PC/BWS annotation with structured output.

Import packages and load API key:

In [ ]:
# import necessary packages
import json
import re

import pandas as pd
from pydantic import BaseModel

from openai import OpenAI

In [ ]:
# load API key and create client

client = OpenAI(
    api_key="" # API key here
)

Read data from local files:
- tweet ids (100) for RS;
- previously generated tuples (150) for PC and BWS
- id-text mapping file, for retrieving tweet texts by ids

In [ ]:
# load id-text mapping and create dictionary

mapping = pd.read_csv('id2text_batch1.csv', dtype={'id': str})
mapping.head()

In [ ]:
# Pilot experiments showed that the model is sometimes inconsistent in keeping/omitting the brackets and quotes around the tweet ids when returning results.
# Therefore, keep only the numeric part of the ids here to avoid confusion:

def normalize_tweet_id(value):
    value = str(value).strip()
    match = re.fullmatch(r"\['(\d+)'\]", value)
    if match:
        return match.group(1)
    return value

mapping['id'] = mapping['id'].apply(normalize_tweet_id)


In [ ]:
# create dictionary for id-text mapping
id2text = dict(zip(mapping['id'], mapping['text']))
id2text

In [ ]:
# read RS ids

with open('rs_batch1.txt', 'r') as f:
    rs_ids = [normalize_tweet_id(line) for line in f.read().splitlines()]

print(rs_ids[:5])
print(len(rs_ids)) #100 tweets

# read PC tuples

pc_tuples = []

with open('pc_batch1.txt', 'r') as f:
    for line in f:
        line = line.strip()
        ids = [normalize_tweet_id(tweet_id) for tweet_id in line.split()]
        pc_tuples.append(tuple(ids))

print(pc_tuples[:5])
print(len(pc_tuples)) #150 tuples(pairs)

# read BWS tuples

bws_tuples = []

with open('bws_batch1.txt', 'r') as f:
    for line in f:
        line = line.strip()
        ids = [normalize_tweet_id(tweet_id) for tweet_id in line.split()]
        bws_tuples.append(tuple(ids))

print(bws_tuples[:5])
print(len(bws_tuples)) #150 tuples

In [ ]:
# select a few examples for pilot testing: here first 5

n_pilot = 5 

rs_pilot = rs_ids[:n_pilot]
pc_pilot = pc_tuples[:n_pilot]
bws_pilot = bws_tuples[:n_pilot]

Building record items (=trials in the original experiment) for prompting:
- item_id (1-100/150) is for identifying the trial.

In [ ]:
# RS

rs_records = []
for idx, tweet_id in enumerate(rs_ids):
    rs_records.append({
        "item_id": f"rs_{idx+1}",
        "tweet_id": tweet_id,
        "text": id2text[tweet_id]
    })

print(rs_records[0])
print(len(rs_records)) #100 records

# PC

pc_records = []
for idx, pair in enumerate(pc_tuples):
    ids = list(pair)
    texts = [id2text[id] for id in ids]
    
    pc_records.append({
        "item_id": f"pc_{idx+1}",
        "tweet_ids": ids,
        "texts": texts
    })

print(pc_records[0])
print(len(pc_records)) #150 records

# BWS

bws_records = []
for idx, bws_tuple in enumerate(bws_tuples):
    ids = list(bws_tuple)
    texts = [id2text[id] for id in ids]

    bws_records.append({
        "item_id": f"bws_{idx+1}",
        "tweet_ids": ids,
        "texts": texts
    })

print(bws_records[0])
print(len(bws_records)) #150 records

Structured output schemas:

In [ ]:
class RSOutput(BaseModel):
    valence: int
    arousal: int
    dominance: int

class PCOutput(BaseModel):
    valence_best: str
    arousal_best: str
    dominance_best: str

class BWSOutput(BaseModel):
    valence_best: str
    valence_worst: str
    arousal_best: str
    arousal_worst: str
    dominance_best: str
    dominance_worst: str

# Add the item(trial) id to the output directly from the record
def attach_item_id(parsed_output, record):
    result = parsed_output.model_dump()
    result['item_id'] = record['item_id']
    return result


Define functions for building prompts based on records:
currently:
- role & task
- introduction VAD (combined and reduced version of that in 1 and 3.4)
- guidelines (as the original study mentioned)
- Task description + tweet(s)

In [ ]:
# RS

def build_rs_prompt(record):
    tweet_text = record["text"]
    prompt = f"""
You are an annotator specialized in annotating affective dimensions in user-generated content. 
You will evaluate a Dutch tweet based on three affective dimensions:
1. Valence: the level of positivity or pleasantness. Low values indicate unhappiness; high values indicate happiness.
2. Arousal: the level of activation or energy. Low values indicate calmness; high values indicate excitement.
3. Dominance: the feeling of control or power. Low values indicate submissiveness; high values indicate dominance.

Guidelines: 
1. Infer the emotional state of the writer when they wrote the tweet, not your own reaction as a reader.
2. Do not rely too much on emojis and explicit emotion words, as they do not always reflect the writer's emotional state, and emotion may be present without explicit expression.

Task:
For the tweet below, assign a score (integer) from 0 to 100 for each of the three dimensions (valence, arousal, dominance).
If the tweet is neutral (no emotion present), assign a score of 50 for all dimensions.

Tweet Text: 
{tweet_text}

Return your answer in the specified structured format. Do not return null values, multiple answers, or any additional text.
"""
    return prompt


In [ ]:
# PC

def build_pc_prompt(record):
    tweet1_id = record["tweet_ids"][0]
    tweet2_id = record["tweet_ids"][1]
    tweet1_text = record["texts"][0]
    tweet2_text = record["texts"][1]
    prompt = f"""
You are an annotator specialized in annotating affective dimensions in user-generated content. 
You will evaluate Dutch tweets based on three affective dimensions:
1. Valence: the level of positivity or pleasantness. Low values indicate unhappiness; high values indicate happiness.
2. Arousal: the level of activation or energy. Low values indicate calmness; high values indicate excitement.
3. Dominance: the feeling of control or power. Low values indicate submissiveness; high values indicate dominance.

Guidelines: 
1. Infer the emotional state of the writer when they wrote the tweet, not your own reaction as a reader.
2. Do not rely too much on emojis and explicit emotion words, as they do not always reflect the writer's emotional state, and emotion may be present without explicit expression.

Task:
For the 2 tweets below, compare them separately for valence, arousal, and dominance. For each dimension, select the tweet with the higher value, and return only its Tweet ID exactly as given.
If the tweet is neutral (no emotion present), treat it as having a medium value for all dimensions.

Tweet 1:
Tweet ID: {tweet1_id}
Tweet Text: {tweet1_text}

Tweet 2:
Tweet ID: {tweet2_id}
Tweet Text: {tweet2_text}

Return your answer in the specified structured format. Do not return null values, multiple answers, or any additional text.
"""
    return prompt


In [ ]:
# BWS
def build_bws_prompt(record):
    tweet1_id = record["tweet_ids"][0]
    tweet2_id = record["tweet_ids"][1]
    tweet3_id = record["tweet_ids"][2]
    tweet4_id = record["tweet_ids"][3]
    tweet1_text = record["texts"][0]
    tweet2_text = record["texts"][1]
    tweet3_text = record["texts"][2]
    tweet4_text = record["texts"][3]
    prompt = f"""
You are an annotator specialized in annotating affective dimensions in user-generated content. 
You will evaluate Dutch tweets based on three affective dimensions:
1. Valence: the level of positivity or pleasantness. Low values indicate unhappiness; high values indicate happiness.
2. Arousal: the level of activation or energy. Low values indicate calmness; high values indicate excitement.
3. Dominance: the feeling of control or power. Low values indicate submissiveness; high values indicate dominance.

Guidelines: 
1. Infer the emotional state of the writer when they wrote the tweet, not your own reaction as a reader.
2. Do not rely too much on emojis and explicit emotion words, as they do not always reflect the writer's emotional state, and emotion may be present without explicit expression.

Task:
For the 4 tweets below, compare them and identify for each of the three dimensions (valence, arousal, dominance):
- The tweet with the highest value (best)
- The tweet with the lowest value (worst)
Return only their Tweet IDs exactly as given.
If the tweet is neutral (no emotion present), treat it as having a medium value for all dimensions.

Tweet 1:
Tweet ID: {tweet1_id}
Tweet Text: {tweet1_text}

Tweet 2:
Tweet ID: {tweet2_id}
Tweet Text: {tweet2_text}

Tweet 3:
Tweet ID: {tweet3_id}
Tweet Text: {tweet3_text}

Tweet 4:
Tweet ID: {tweet4_id}
Tweet Text: {tweet4_text}

Return your answer in the specified structured format. Do not return null values, multiple answers, or any additional text.
"""
    return prompt

Test prompt construction:
- Generate one example prompt for each method with the functions above

In [ ]:
print("======= RS Prompt 1: =======")
print(build_rs_prompt(rs_records[0]))

print("\n======= PC Prompt 1: =======")
print(build_pc_prompt(pc_records[0]))

print("\n======= BWS Prompt 1: =======")
print(build_bws_prompt(bws_records[0]))

Define functions for making API calls for annotating:

In [ ]:
# the LLM annotator:

MODEL = "gpt-5.4" # change each time when testing different models

# RS

def annotate_rs(record, model=MODEL):
    prompt = build_rs_prompt(record)
    
    response = client.responses.parse(
        model=model,
        input=prompt,
        text_format=RSOutput,
        reasoning={"effort":"low"}
    )
    
    parsed = response.output_parsed
    return attach_item_id(parsed, record)

# PC

def annotate_pc(record, model=MODEL):
    prompt = build_pc_prompt(record)

    response = client.responses.parse(
        model=model,
        input=prompt,
        text_format=PCOutput,
        reasoning={"effort":"low"}
    )

    parsed = response.output_parsed
    return attach_item_id(parsed, record)

# BWS

def annotate_bws(record, model=MODEL):
    prompt = build_bws_prompt(record)

    response = client.responses.parse(
        model=model,
        input=prompt,
        text_format=BWSOutput,
        reasoning={"effort":"low"}
    )

    parsed = response.output_parsed
    return attach_item_id(parsed, record)

Single item test:

In [ ]:
rs_test = annotate_rs(rs_records[0])
pc_test = annotate_pc(pc_records[0])
bws_test = annotate_bws(bws_records[0])

print("RS test result:")
print(rs_test)

print("\nPC test result:")
print(pc_test)

print("\nBWS test result:")
print(bws_test)

Full experiment:

In [ ]:
rs_results = [annotate_rs(record) for record in rs_records]
pc_results = [annotate_pc(record) for record in pc_records]
bws_results = [annotate_bws(record) for record in bws_records]

Save output as JSON files:

In [ ]:
output_files = {
    'rs_outputs.json': rs_results,
    'pc_outputs.json': pc_results,
    'bws_outputs.json': bws_results,
}

for filename, results in output_files.items():
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

In [ ]:
# save records (item_id - tweet_id(s) mapping) for preparing files for score conversion:

rs_map = [{"item_id": r["item_id"], "tweet_id": r["tweet_id"]} for r in rs_records]
pc_map = [{"item_id": r["item_id"], "tweet_ids": r["tweet_ids"]} for r in pc_records]
bws_map = [{"item_id": r["item_id"], "tweet_ids": r["tweet_ids"]} for r in bws_records]

# DataFrames for inspection / CSV export (convert lists to space-separated strings for CSV)
df_rs = pd.DataFrame(rs_map)
df_pc = pd.DataFrame(pc_map)
df_bws = pd.DataFrame(bws_map)

# for PC and BWS, split the 2 or 4 tweet ids into separate columns
df_pc_csv = df_pc.copy()
df_pc_csv[["tweet_id_1", "tweet_id_2"]] = pd.DataFrame(df_pc_csv["tweet_ids"].tolist(), index=df_pc_csv.index)
df_pc_csv.drop(columns=["tweet_ids"], inplace=True)

df_bws_csv = df_bws.copy()
df_bws_csv[["tweet_id_1", "tweet_id_2", "tweet_id_3", "tweet_id_4"]] = pd.DataFrame(df_bws_csv["tweet_ids"].tolist(), index=df_bws_csv.index)
df_bws_csv.drop(columns=["tweet_ids"], inplace=True)

# Save files
df_rs.to_csv("rs_itemid_tweetid.csv", index=False)
df_pc_csv.to_csv("pc_itemid_tweetids.csv", index=False)
df_bws_csv.to_csv("bws_itemid_tweetids.csv", index=False)

In addition: a coarser-grained rating scale (0-10)
- from reading ids and building records onwards

In [ ]:
# read RS ids

with open('rs_batch1.txt', 'r') as f:
    rs_ids = [normalize_tweet_id(line) for line in f.read().splitlines()]

print(rs_ids[:5])
print(len(rs_ids)) #100 tweets

# RS

rs_records = []
for idx, tweet_id in enumerate(rs_ids):
    rs_records.append({
        "item_id": f"rs_{idx+1}",
        "tweet_id": tweet_id,
        "text": id2text[tweet_id]
    })

print(rs_records[0])
print(len(rs_records)) #100 records

# Structured output

class RSOutput(BaseModel):
    valence: int
    arousal: int
    dominance: int


# Add the item(trial) id to the output directly from the record
def attach_item_id(parsed_output, record):
    result = parsed_output.model_dump()
    result['item_id'] = record['item_id']
    return result

# Prompt

def build_rs10_prompt(record):
    tweet_text = record["text"]
    prompt = f"""
You are an annotator specialized in annotating affective dimensions in user-generated content. 
You will evaluate a Dutch tweet based on three affective dimensions:
1. Valence: the level of positivity or pleasantness. Low values indicate unhappiness; high values indicate happiness.
2. Arousal: the level of activation or energy. Low values indicate calmness; high values indicate excitement.
3. Dominance: the feeling of control or power. Low values indicate submissiveness; high values indicate dominance.

Guidelines: 
1. Infer the emotional state of the writer when they wrote the tweet, not your own reaction as a reader.
2. Do not rely too much on emojis and explicit emotion words, as they do not always reflect the writer's emotional state, and emotion may be present without explicit expression.

Task:
For the tweet below, assign a score (integer) from 0 to 10 for each of the three dimensions (valence, arousal, dominance).
If the tweet is neutral (no emotion present), assign a score of 5 for all dimensions.

Tweet Text: 
{tweet_text}

Return your answer in the specified structured format. Do not return null values, multiple answers, or any additional text.
"""
    return prompt

print("======= RS10 Prompt 1: =======")
print(build_rs10_prompt(rs_records[0]))

In [ ]:
# the LLM annotator:

MODEL = "gpt-5.4" # change each time when testing different models

# RS10

def annotate_rs10(record, model=MODEL):
    prompt = build_rs10_prompt(record)

    response = client.responses.parse(
        model=model,
        input=prompt,
        text_format=RSOutput,
        reasoning={"effort":"low"}
    )
    
    parsed = response.output_parsed
    return attach_item_id(parsed, record)

In [ ]:
# single item test

rs10_test = annotate_rs10(rs_records[0])
print("RS10 test result:")
print(rs10_test)

In [ ]:
rs10_results = [annotate_rs10(record) for record in rs_records]
output_file = f'{MODEL}_rs10_outputs.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(rs10_results, f, indent=2, ensure_ascii=False)